# Домашнее задание 2. Анализ данных о пассажирах Титаника

Работаем с файлом `train.csv` из соревнования Kaggle «Titanic». Каждая строка — один пассажир.

Основные столбцы:

- `Survived` — выжил ли пассажир (1 — да, 0 — нет);
- `Pclass` — класс билета (1, 2, 3);
- `Name`, `Sex`, `Age` — имя, пол, возраст;
- `SibSp` — число братьев, сестёр и супругов на борту;
- `Parch` — число родителей и детей на борту;
- `Ticket`, `Fare`, `Cabin`, `Embarked` — билет, стоимость, каюта, порт посадки.

Файл `train.csv` лежит в той же папке, что и ноутбук.

## 1. Чтение датасета

Подключаем библиотеку `pandas` и читаем CSV-файл функцией `pd.read_csv`. Результат — таблица `DataFrame`. Метод `head()` показывает первые строки.

In [1]:
import re

import pandas as pd

df = pd.read_csv("train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## 2. Основная информация о датасете

Размер таблицы: атрибут `shape` возвращает пару (число строк, число столбцов).

In [2]:
df.shape

(891, 12)

Метод `info()` показывает типы данных столбцов и количество непустых значений в каждом.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


Число пропусков в каждом столбце: `isna()` отмечает пустые ячейки, `sum()` считает их по столбцам.

In [4]:
df.isna().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

Метод `describe()` выводит статистику по числовым столбцам: количество, среднее (`mean`), стандартное отклонение, минимум, квартили и максимум.

In [5]:
df.describe()

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


Отдельно выведем средние значения числовых столбцов.

In [6]:
df.mean(numeric_only=True)

PassengerId    446.000000
Survived         0.383838
Pclass           2.308642
Age             29.699118
SibSp            0.523008
Parch            0.381594
Fare            32.204208
dtype: float64

Для нечисловых (текстовых) столбцов `describe(exclude="number")` показывает число уникальных значений, самое частое значение (`top`) и его частоту (`freq`).

In [7]:
df.describe(exclude="number")

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


**Выводы:**

- В датасете 891 пассажир и 12 столбцов.
- Пропуски есть в трёх столбцах: `Age` (177), `Cabin` (687, то есть каюта известна менее чем у четверти пассажиров) и `Embarked` (2).
- Среднее значение `Survived` около 0.38: выжило примерно 38 % пассажиров.
- Средний возраст около 29.7 лет, средняя стоимость билета около 32.2.

## 3. Процент выживаемости по классам

Группируем пассажиров по `Pclass` методом `groupby`. Столбец `Survived` содержит 0 и 1, поэтому его среднее в группе — это доля выживших. Умножаем на 100, чтобы получить проценты, и округляем до одного знака.

In [8]:
survival_by_class = (df.groupby("Pclass")["Survived"].mean() * 100).round(1)
survival_by_class

Pclass
1    63.0
2    47.3
3    24.2
Name: Survived, dtype: float64

**Вывод:** чем выше класс, тем выше шанс выжить. В первом классе выжило около 63 % пассажиров, во втором около 47 %, в третьем только около 24 %.

## 4. Самое популярное мужское и женское имя

### Как достать имя из столбца `Name`

Имена записаны в формате `Фамилия, Обращение. Имя Второе_имя`, например `Braund, Mr. Owen Harris`. Нужное нам имя идёт сразу после обращения (`Mr.`, `Miss.`, `Master.` и т. д.).

Есть особые случаи:

- У замужних женщин (`Mrs.`) после обращения стоит имя **мужа**, а собственное имя записано в скобках: `Cumings, Mrs. John Bradley (Florence Briggs Thayer)`. Здесь нужно имя `Florence`, а не `John`. Так же устроены записи с обращениями `Mme`, `Lady` и `the Countess`.
- Иногда после обращения сразу идут скобки: `Meanwell, Miss. (Marion Ogden)`. Тогда имя тоже берём из скобок.
- В кавычках указаны прозвища или другие имена: `Nakid, Miss. Maria ("Mary")`. Их убираем.
- Встречается запись через слеш: `Carl/Charles`. Берём первое имя.

Функция `get_first_name` учитывает эти случаи:

1. отделяет обращение от остальной части имени;
2. удаляет текст в кавычках с помощью регулярного выражения;
3. выбирает, откуда брать имя: из скобок или из текста перед ними;
4. возвращает первое слово.

In [9]:
MARRIED_TITLES = {"Mrs", "Mme", "Lady", "the Countess"}


def get_first_name(full_name: str) -> str | None:
    after_surname = full_name.split(",", 1)[1]
    title, rest = after_surname.split(".", 1)
    title = title.strip()

    rest = re.sub(r'"[^"]*"', "", rest)

    before_brackets = rest.split("(", 1)[0]
    if "(" in rest and (title in MARRIED_TITLES or before_brackets.strip() == ""):
        rest = rest.split("(", 1)[1]
    else:
        rest = before_brackets

    words = re.findall(r"[A-Za-z']+", rest)
    return words[0] if words else None

Проверим функцию на нескольких сложных записях.

In [10]:
examples = [
    "Braund, Mr. Owen Harris",
    "Cumings, Mrs. John Bradley (Florence Briggs Thayer)",
    "Meanwell, Miss. (Marion Ogden)",
    'Nakid, Miss. Maria ("Mary")',
    "Widegren, Mr. Carl/Charles Peter",
    "Rothes, the Countess. of (Lucy Noel Martha Dyer-Edwards)",
]

for name in examples:
    print(f"{name}  ->  {get_first_name(name)}")

Braund, Mr. Owen Harris  ->  Owen
Cumings, Mrs. John Bradley (Florence Briggs Thayer)  ->  Florence
Meanwell, Miss. (Marion Ogden)  ->  Marion
Nakid, Miss. Maria ("Mary")  ->  Maria
Widegren, Mr. Carl/Charles Peter  ->  Carl
Rothes, the Countess. of (Lucy Noel Martha Dyer-Edwards)  ->  Lucy


Применяем функцию ко всему столбцу `Name` методом `apply` и сохраняем результат в новый столбец `FirstName`. Затем проверяем, что имя удалось извлечь у всех пассажиров.

In [11]:
df["FirstName"] = df["Name"].apply(get_first_name)

print("Пассажиров без извлечённого имени:", df["FirstName"].isna().sum())
df[["Name", "Sex", "FirstName"]].head(10)

Пассажиров без извлечённого имени: 0


,Name,Sex,FirstName
0,"Braund, Mr. Owen Harris",male,Owen
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,Florence
2,"Heikkinen, Miss. Laina",female,Laina
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,Lily
4,"Allen, Mr. William Henry",male,William
5,"Moran, Mr. James",male,James
6,"McCarthy, Mr. Timothy J",male,Timothy
7,"Palsson, Master. Gosta Leonard",male,Gosta
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,Elisabeth
9,"Nasser, Mrs. Nicholas (Adele Achem)",female,Adele


### Самые популярные имена

Сначала посмотрим на топ-5 имён для каждого пола. `value_counts()` считает, сколько раз встречается каждое имя.

In [12]:
for sex in ["male", "female"]:
    print(f"Топ-5 имён ({sex}):")
    print(df.loc[df["Sex"] == sex, "FirstName"].value_counts().head(5))
    print()

Топ-5 имён (male):
FirstName
William    35
John       25
George     14
Charles    13
Thomas     13
Name: count, dtype: int64

Топ-5 имён (female):
FirstName
Anna         15
Mary         14
Elizabeth    11
Margaret     10
Alice         7
Name: count, dtype: int64



Самое частое значение находит метод `mode()`. Если несколько имён встречаются одинаково часто, `mode()` вернёт их все, поэтому соединяем результат в строку через запятую. Рядом выводим, сколько раз встречается это имя.

In [13]:
def most_popular(names: pd.Series) -> str:
    return ", ".join(names.mode())


def top_count(names: pd.Series) -> int:
    return names.value_counts().max()


popular_names = df.groupby("Sex")["FirstName"].agg(name=most_popular, count=top_count)
popular_names

,name,count
Sex,,
female,Anna,15
male,William,35


**Вывод:** самое популярное мужское имя — **William**, самое популярное женское — **Anna**.

Если брать у замужних женщин имя из основной части записи, а не из скобок, результат получится неверным: вместо имён женщин считались бы имена их мужей.

## 5. Самые популярные имена в каждом классе

Используем те же функции, но группируем сразу по двум столбцам: `Pclass` и `Sex`. Метод `unstack()` разворачивает пол в столбцы, чтобы таблицу было удобно читать.

In [14]:
popular_by_class = df.groupby(["Pclass", "Sex"])["FirstName"].agg(
    name=most_popular, count=top_count
)
popular_by_class.unstack("Sex")

name           count     
Sex                  female     male female male
Pclass                                          
1       Elizabeth, Margaret  William      5   11
2                 Elizabeth  William      5    9
3                      Anna  William      9   15

**Вывод:**

- Среди мужчин во всех трёх классах чаще всего встречается имя **William**.
- Среди женщин: в первом классе поровну **Elizabeth** и **Margaret**, во втором **Elizabeth**, в третьем **Anna**.

В первом классе у женщин ничья, поэтому выводятся оба имени.

## 6. Пассажиры старше 44 лет

Фильтруем таблицу по условию `df["Age"] > 44`. Результат условия — столбец из `True` и `False`; в квадратных скобках он оставляет только строки со значением `True`.

Пассажиры с неизвестным возрастом (`NaN`) в выборку не попадают: сравнение `NaN > 44` даёт `False`.

In [15]:
older_44 = df[df["Age"] > 44]

print("Количество пассажиров старше 44 лет:", len(older_44))
older_44.head(10)

Количество пассажиров старше 44 лет: 115


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
6,7,0,1,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S,Timothy
11,12,1,1,"Bonnell, Miss. Elizabeth",female,58.0,0,0,113783,26.5500,C103,S,Elizabeth
15,16,1,2,"Hewlett, Mrs. (Mary D Kingcome)",female,55.0,0,0,248706,16.0000,NaN,S,Mary
33,34,0,2,"Wheadon, Mr. Edward H",male,66.0,0,0,C.A. 24579,10.5000,NaN,S,Edward
52,53,1,1,"Harper, Mrs. Henry Sleeper (Myna Haxtun)",female,49.0,1,0,PC 17572,76.7292,D33,C,Myna
54,55,0,1,"Ostby, Mr. Engelhart Cornelius",male,65.0,0,1,113509,61.9792,B30,C,Engelhart
62,63,0,1,"Harris, Mr. Henry Birkhardt",male,45.0,1,0,36973,83.4750,C83,S,Henry
92,93,0,1,"Chaffee, Mr. Herbert Fuller",male,46.0,1,0,W.E.P. 5734,61.1750,E31,S,Herbert
94,95,0,3,"Coxon, Mr. Daniel",male,59.0,0,0,364500,7.2500,NaN,S,Daniel
96,97,0,1,"Goldschmidt, Mr. George B",male,71.0,0,0,PC 17754,34.6542,A5,C,George


Посмотрим, кто входит в эту группу и какая в ней доля выживших.

In [16]:
print("По полу:", older_44["Sex"].value_counts().to_dict())
print(f"Доля выживших: {older_44['Survived'].mean() * 100:.1f} %")
print(f"Доля выживших среди всех пассажиров: {df['Survived'].mean() * 100:.1f} %")

По полу: {'male': 79, 'female': 36}
Доля выживших: 37.4 %
Доля выживших среди всех пассажиров: 38.4 %


**Вывод:** возраст больше 44 лет указан у 115 пассажиров, из них 79 мужчин и 36 женщин. Выжило около 37 % из них, это почти столько же, сколько по всему кораблю (около 38 %).

## 7. Мужчины младше 44 лет

Объединяем два условия оператором `&` (логическое «И»). Каждое условие обязательно берём в круглые скобки: у `&` приоритет выше, чем у операторов сравнения.

In [17]:
young_men = df[(df["Age"] < 44) & (df["Sex"] == "male")]

print("Количество мужчин младше 44 лет:", len(young_men))
young_men.head(10)

Количество мужчин младше 44 лет: 368


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,FirstName
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.250,NaN,S,Owen
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.050,NaN,S,William
7,8,0,3,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.075,NaN,S,Gosta
12,13,0,3,"Saundercock, Mr. William Henry",male,20.0,0,0,A/5. 2151,8.050,NaN,S,William
13,14,0,3,"Andersson, Mr. Anders Johan",male,39.0,1,5,347082,31.275,NaN,S,Anders
16,17,0,3,"Rice, Master. Eugene",male,2.0,4,1,382652,29.125,NaN,Q,Eugene
20,21,0,2,"Fynney, Mr. Joseph J",male,35.0,0,0,239865,26.000,NaN,S,Joseph
21,22,1,2,"Beesley, Mr. Lawrence",male,34.0,0,0,248698,13.000,D56,S,Lawrence
23,24,1,1,"Sloper, Mr. William Thompson",male,28.0,0,0,113788,35.500,A6,S,William
27,28,0,1,"Fortune, Mr. Charles Alexander",male,19.0,3,2,19950,263.000,C23 C25 C27,S,Charles


Сравним долю выживших в этой группе с долей среди всех пассажиров и среди женщин.

In [18]:
print("По классам:", young_men["Pclass"].value_counts().sort_index().to_dict())
print(f"Доля выживших среди мужчин младше 44 лет: {young_men['Survived'].mean() * 100:.1f} %")
print(f"Доля выживших среди всех пассажиров: {df['Survived'].mean() * 100:.1f} %")
print(f"Доля выживших среди женщин: {df.loc[df['Sex'] == 'female', 'Survived'].mean() * 100:.1f} %")

По классам: {1: 54, 2: 82, 3: 232}
Доля выживших среди мужчин младше 44 лет: 21.2 %
Доля выживших среди всех пассажиров: 38.4 %
Доля выживших среди женщин: 74.2 %


**Вывод:** в выборку попали 368 мужчин младше 44 лет, больше половины из них (232) ехали третьим классом. Выжил только около 21 % из них, заметно меньше, чем в среднем по кораблю (около 38 %) и среди женщин (около 74 %). Это соответствует правилу «сначала женщины и дети» при посадке в шлюпки.

В выборку попали и мальчики: у них обращение `Master`, но пол тоже `male`.

## 8. Количество n-местных кают

1. `value_counts()` по столбцу `Cabin` показывает, сколько пассажиров записано в каждую каюту. Пустые значения при этом отбрасываются.
2. Второй `value_counts()` по полученным числам показывает, сколько кают с одним, двумя, тремя и т. д. пассажирами.
3. По заданию нужны каюты, в которых было 2 человека и больше, поэтому отбираем их отдельно.

In [19]:
people_per_cabin = df["Cabin"].value_counts()
people_per_cabin.head(10)

Cabin
G6             4
C23 C25 C27    4
B96 B98        4
F33            3
E101           3
F2             3
D              3
C22 C26        3
C123           2
D33            2
Name: count, dtype: int64

In [20]:
cabins_by_size = people_per_cabin.value_counts().sort_index()
cabins_by_size.index.name = "Человек в каюте"
cabins_by_size.name = "Количество кают"
cabins_by_size

Человек в каюте
1    101
2     38
3      5
4      3
Name: Количество кают, dtype: int64

In [21]:
cabins_by_size[cabins_by_size.index >= 2]

Человек в каюте
2    38
3     5
4     3
Name: Количество кают, dtype: int64

**Вывод:** 2-местных кают 38, 3-местных 5, 4-местных 3. Ещё 101 каюта занята одним пассажиром.

Ограничения подсчёта:

- В файле `train.csv` только часть пассажиров Титаника, а каюта известна лишь у 204 из них. На самом деле в каютах могло быть больше людей.
- Иногда в `Cabin` записано несколько кают сразу (например, `C23 C25 C27`). Такая запись считается одной «каютой», так как это одно бронирование.

## 9. Пассажиры без родственников на борту

Пассажир путешествует без родственников, если у него нет ни братьев, сестёр и супругов (`SibSp == 0`), ни родителей и детей (`Parch == 0`). Отбираем такие строки и считаем их количество. Сумма столбца из `True`/`False` равна числу значений `True`.

In [22]:
alone = (df["SibSp"] == 0) & (df["Parch"] == 0)

print("Пассажиров без родственников на борту:", alone.sum())
print(f"Это {alone.mean() * 100:.1f} % от всех пассажиров")

Пассажиров без родственников на борту: 537
Это 60.3 % от всех пассажиров


**Вывод:** больше половины пассажиров (537 человек, около 60 %) путешествовали без родственников.